In [1]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax


In [2]:
import google.protobuf
print("protobuf installed successfully")


protobuf installed successfully


In [3]:
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

sentiment_task = pipeline(
    "sentiment-analysis",
    model=model_path,
    tokenizer=model_path
)

sentiment_task("T'estimo!")


Device set to use cpu


[{'label': 'positive', 'score': 0.6600585579872131}]

In [4]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)


In [5]:
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)

model = AutoModelForSequenceClassification.from_pretrained(MODEL)


In [7]:
text = "Good night 😊"
text = preprocess(text)

print("Processed text:", text)


Processed text: Good night 😊


In [8]:
encoded_input = tokenizer(text, return_tensors='pt')
encoded_input


{'input_ids': tensor([[    0, 18621, 17431,     6, 82803,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [9]:
output = model(**encoded_input)


In [10]:
scores = output[0][0].detach().numpy()
scores = softmax(scores)

scores


array([0.03125937, 0.20148005, 0.7672607 ], dtype=float32)

In [11]:
ranking = np.argsort(scores)[::-1]

for i in range(scores.shape[0]):
    label = config.id2label[ranking[i]]
    score = scores[ranking[i]]
    print(f"{i+1}) {label} {np.round(float(score), 4)}")


1) positive 0.7673
2) neutral 0.2015
3) negative 0.0313


In [13]:
import pandas as pd

df = pd.read_csv("../datasets/custom_sentiment_dataset.csv")
df


,id,text,language_type,gold_label
0,1,Good night 😊,English,positive
1,2,I am feeling very sad today,English,negative
2,3,The weather is okay,English,neutral
3,4,শুভ রাত্রি 😊,Bengali,positive
4,5,আজ মনটা খুব খারাপ,Bengali,negative
5,6,আজ আবহাওয়া মোটামুটি,Bengali,neutral
6,7,Good না লাগতেছে আজ,CodeMixed,negative
7,8,আজ presentation ta ভালো হয়েছে 😊,CodeMixed,positive
8,9,Exam টা okay ছিল,CodeMixed,neutral
9,10,I love বাংলাদেশের মানুষ ❤️,CodeMixed,positive


# **BanglaBook Dataset Test with XMLr roBERTa**

In [14]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [15]:
banglabook = pd.read_csv("../datasets/banglabook/csv/test.csv")

print(banglabook.head())


       id                                          Book_Name  \
0   72742   কম্পিউটার প্রোগ্রামিং ৩য় খণ্ড : ডেটা স্ট্রাকচ...   
1  100537   ম্যাসেজ (হার্ডকভার)  আধুনিক মননে দ্বীনের ছোঁয়...   
2   10441   মুক্তিযুদ্ধ নিয়ে স্মৃতিচারণমূলক ৫টি বই (হার্ড...   
3  126428                                      Bandhobi        
4   54626   A Clash of Kings (Book 2 Of A Song Of Ice And...   

              Writer_Name                                   Category  Rating  \
0   তামিম শাহরিয়ার সুবিন                       প্রোগ্রামিং বেসিক বই        1   
1   মিজানুর রহমান আজহারি                      ইসলামি আদর্শ  ও মতবাদ        5   
2        হাসান আজিজুল হক    মুক্তিযুদ্ধের ডায়েরি, চিঠি ও স্মৃতিচারণ        5   
3              Raba Khan                              English Story        1   
4    George R. R. Martin                    Novel: English Language        5   

                                              Review      Site sentiment  \
0  দারুন ‍একটা বই ,প্রথমে ভেবেছিলাম  Online থেকে ...  Roko

In [25]:
print(banglabook.columns)


Index(['id', 'Book_Name', 'Writer_Name', 'Category', 'Rating', 'Review',
       'Site', 'sentiment', 'label'],
      dtype='object')


In [16]:
banglabook = banglabook.rename(columns={
    "Review": "text",
    "label": "gold_label"
})


In [17]:
def normalize_label(label):
    if label == 0:
        return "negative"
    elif label == 1:
        return "neutral"
    elif label == 2:
        return "positive"
    else:
        return "neutral"

banglabook["gold_label"] = banglabook["gold_label"].apply(normalize_label)


In [18]:
banglabook = banglabook.dropna(subset=["text"])
banglabook["text"] = banglabook["text"].astype(str)


In [19]:
def predict_label(text):
    if not isinstance(text, str) or text.strip() == "":
        return "neutral"

    result = sentiment_task(text)[0]   # single dict
    return result["label"]


In [20]:
tqdm.pandas()

banglabook["predicted_label"] = banglabook["text"].progress_apply(predict_label)



100%|██████████| 31614/31614 [1:07:38<00:00,  7.79it/s]


In [21]:
from sklearn.metrics import accuracy_score, classification_report

y_true = banglabook["gold_label"]
y_pred = banglabook["predicted_label"]

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


Accuracy: 0.5514329094704877
              precision    recall  f1-score   support

    negative       0.22      0.41      0.29      1935
     neutral       0.05      0.41      0.09      1361
    positive       0.95      0.57      0.71     28318

    accuracy                           0.55     31614
   macro avg       0.41      0.46      0.36     31614
weighted avg       0.86      0.55      0.66     31614

